In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
from pathlib import Path

# Base input directory
input_dir = Path('/kaggle/input')

# 1. Recursively find the main data folders
# This handles the nesting shown in your 'datasets' folder
pneu_root = next(input_dir.rglob('chest_xray'))
tb_root = next(input_dir.rglob('TB_Chest_Radiography_Database'))

print(f"Found Pneumonia path: {pneu_root}")
print(f"Found TB path: {tb_root}")

# 2. Gather the paths for your 3-class system: Normal, Pneumonia, Tuberculosis
# Combining Normal images from BOTH datasets to fix the 0.38 recall issue 
normal_paths = list((pneu_root / 'train' / 'NORMAL').glob('*.jpeg')) + \
               list((pneu_root / 'test' / 'NORMAL').glob('*.jpeg')) + \
               list((pneu_root / 'val' / 'NORMAL').glob('*.jpeg')) + \
               list((tb_root / 'Normal').glob('*.png'))

# Pneumonia images (Train + Test for a larger pool) [cite: 145]
pneumonia_paths = list((pneu_root / 'train' / 'PNEUMONIA').glob('*.jpeg')) + \
                  list((pneu_root / 'val' / 'PNEUMONIA').glob('*.jpeg')) + \
                  list((pneu_root / 'test' / 'PNEUMONIA').glob('*.jpeg'))

# Tuberculosis images [cite: 46, 48]
tb_paths = list((tb_root / 'Tuberculosis').glob('*.png'))

print(f"\n--- Unified Class Distribution ---")
print(f"Class: Normal       | Count: {len(normal_paths)}")
print(f"Class: Pneumonia    | Count: {len(pneumonia_paths)}")
print(f"Class: Tuberculosis | Count: {len(tb_paths)}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Creating the master list for your 3-class target
data = []
for p in normal_paths: data.append({'path': str(p), 'label': 'normal'})
for p in pneumonia_paths: data.append({'path': str(p), 'label': 'pneumonia'})
for p in tb_paths: data.append({'path': str(p), 'label': 'tuberculosis'})

df = pd.DataFrame(data)

# Stratified Split: Ensuring the 700 TB images are balanced across sets
# This addresses the "Focus on lung region" and "Add more data" roadmap goals
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
import torch
from torchvision import transforms
import cv2
import numpy as np
from PIL import Image

# 1. Your Research-Optimized CLAHE
class CLAHETransform:
    def __init__(self, clip_limit=1.5, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img)
        if len(img_np.shape) == 2: 
            img_np = self.clahe.apply(img_np)
        else:
            img_lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            img_lab[:,:,0] = self.clahe.apply(img_lab[:,:,0])
            img_np = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(img_np)

# 2. Training Pipeline (The "Secret Sauce" to prevent overfitting)
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)), 
    CLAHETransform(clip_limit=2.0), 
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Validation/Test Pipeline (Fixed name to resolve NameError)
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Matches the required input for Swin Transformer
    CLAHETransform(clip_limit=1.5), # Milder CLAHE for stable evaluation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transformation pipelines (including val_test_transforms) are now defined.")

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
class LungDataset(Dataset):
    def __init__(self, df, transform=None, use_masking=False):
        self.df = df
        self.transform = transform
        self.use_masking = use_masking
        self.label_map = {'normal': 0, 'pneumonia': 1, 'tuberculosis': 2}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            # 1. Load and convert to RGB (Required for Swin Transformer teacher) [cite: 41, 43]
            image = Image.open(row['path']).convert('RGB')
            
            # 2. Optional Masking (Roadmap goal: Focus on lung region) 
            if self.use_masking:
                # This would be used once you generate masks for the new datasets
                pass 
                
            # 3. Apply your Research Transforms (CLAHE + Secret Sauce)
            if self.transform:
                image = self.transform(image)
                
            label = self.label_map[row['label']]
            return image, torch.tensor(label, dtype=torch.long)
            
        except Exception as e:
            print(f"⚠️ Error loading {row['path']}: {e}")
            return torch.zeros((3, 224, 224)), torch.tensor(0)

In [ ]:
# Initialize Datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
train_ds = LungDataset(train_df, train_transforms)
val_ds = LungDataset(val_df, val_test_transforms)
test_ds = LungDataset(test_df, val_test_transforms)

# Recalculate weights for the Sampler
class_counts = train_df['label'].value_counts()
labels = train_df['label'].map(train_ds.label_map).values
weights = 1. / np.array([class_counts[l] for l in ['normal', 'pneumonia', 'tuberculosis']])
sample_weights = np.array([weights[l] for l in labels])

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

# Final Loaders
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"🚀 DataLoaders ready for training on {len(train_df)} samples.")

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch

# 1. DenseNet121 Teacher (Baseline & Feature Extractor)
# Listed in your status report as a primary baseline model [cite: 32]
class DenseNetTeacher(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        # DenseNet is excellent for identifying small-scale lung consolidations
        self.model = models.densenet121(weights='DenseNet121_Weights.IMAGENET1K_V1')
        self.model.classifier = nn.Linear(self.model.classifier.in_features, num_classes)
        
    def forward(self, x):
        features = self.model.features(x)
        out = nn.ReLU(inplace=True)(features)
        out = nn.AdaptiveAvgPool2d((1, 1))(out)
        out = torch.flatten(out, 1)
        logits = self.model.classifier(out)
        return logits, features # Features for alignment loss

# 2. Swin Transformer Teacher (Primary Framework Teacher)
# Your framework's designated high-capacity teacher [cite: 41]
class SwinTeacher(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = models.swin_t(weights='Swin_T_Weights.IMAGENET1K_V1')
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)
        
    def forward(self, x):
        features = self.model.features(x)
        x = self.model.norm(features)
        x = self.model.permute(x)        
        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.model.head(x)
        return logits, features

# 3. EfficientNet-lite Student (The Final Deployment Model)
# Targeted for lightweight, explainable output [cite: 50, 149]
class StudentEfficientNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = models.efficientnet_b0(weights='EfficientNet_B0_Weights.IMAGENET1K_V1')
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)
        
    def forward(self, x):
        features = self.model.features(x)
        avg_features = self.model.avgpool(features)
        avg_features = torch.flatten(avg_features, 1)
        logits = self.model.classifier(avg_features)
        return logits, features

# --- INITIALIZATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

teacher_dense = DenseNetTeacher(num_classes=3).to(device)
teacher_swin = SwinTeacher(num_classes=3).to(device)
student_model = StudentEfficientNet(num_classes=3).to(device)

# Freeze both teachers to focus GPU resources on the student training [cite: 167]
teacher_dense.eval()
teacher_swin.eval()
for param in list(teacher_dense.parameters()) + list(teacher_swin.parameters()):
    param.requires_grad = False

print("✅ Research-Grade Baseline (DenseNet) and Framework (Swin) models ready.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. THE FOCAL LOSS CLASS
# This focuses training on "hard" samples (like misclassified Normal cases) 
# and rare classes (Tuberculosis) to hit your 90%+ target[cite: 16, 17].
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=3.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

# 2. THE MULTI-TEACHER DISTILLATION LOSS CLASS
# This implements the Proposed Framework: Loss = α·KD + β·CE[cite: 54].
class MultiTeacherDistillLoss(nn.Module):
    def __init__(self, alpha=0.4, T=3.0, device='cuda'):
        super().__init__()
        self.alpha = alpha
        self.T = T
        
        # [Normal, Pneumonia, Tuberculosis]
        # Weighting [1.0, 1.8, 5.0] prioritizes the harder TB and Pneumonia classes.
        weights = torch.tensor([1.0, 1.8, 5.0]).to(device)
        self.focal_loss = FocalLoss(alpha=weights, gamma=3.0)

        # Projections to align Teacher features with Student's 1280 dimensions.
        self.proj_dense = nn.Linear(1024, 1280).to(device)
        self.proj_swin = nn.Linear(768, 1280).to(device)

    # Signature updated to match your keyword arguments in the training loop
    def forward(self, s_logits, labels, t_logits_list, s_features=None, t_features_list=None):
        # 1. Focal Loss (Supervised Knowledge)
        loss_focal = self.focal_loss(s_logits, labels)
        
        # 2. Logit Distillation (Soft Knowledge from Swin & DenseNet) [cite: 32, 41, 50]
        loss_distill = 0
        for t_logits in t_logits_list:
            soft_targets = F.softmax(t_logits / self.T, dim=1)
            soft_logs = F.log_softmax(s_logits / self.T, dim=1)
            loss_distill += F.kl_div(soft_logs, soft_targets, reduction='batchmean') * (self.T**2)
        loss_distill /= len(t_logits_list)
        
        # 3. Feature Alignment (Anatomical Landmark Alignment)
        loss_align = 0
        if s_features is not None and t_features_list is not None:
            s_gap = F.adaptive_avg_pool2d(s_features, (1, 1)).flatten(1)
            
            # Align Teacher 1: DenseNet (NCHW) [cite: 32]
            t1_gap = F.adaptive_avg_pool2d(t_features_list[0], (1, 1)).flatten(1)
            t1_proj = self.proj_dense(t1_gap) 
            loss_align += 1 - F.cosine_similarity(s_gap, t1_proj).mean()
            
            # Align Teacher 2: Swin (NHWC) [cite: 41]
            t2_gap = t_features_list[1].mean(dim=[1, 2]) 
            t2_proj = self.proj_swin(t2_gap) 
            loss_align += 1 - F.cosine_similarity(s_gap, t2_proj).mean()
            
            loss_align /= 2
            
        # Final weighted combination for high-precision detection
        return (0.4 * loss_distill) + (0.4 * loss_focal) + (0.2 * loss_align)

# RE-INITIALIZE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = MultiTeacherDistillLoss(alpha=0.4, T=3.0, device=device)
print("✅ FocalLoss and MultiTeacherDistillLoss are fully integrated.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F

def validate(model, loader, device):
    model.eval() 
    all_preds = []
    all_labels = []
    val_loss = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images) 
            
            loss = F.cross_entropy(logits, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(logits, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_loss = val_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    # Using 'macro' to ensure each class (Normal, Pneumonia, TB) is weighted equally 
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return avg_loss, acc, f1, rec

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
num_epochs = 25
patience = 4  # Early stopping to prevent overfitting [cite: 166]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Optimizer & High-Performance Scheduler
optimizer = optim.Adam(student_model.parameters(), lr=1e-4)

scheduler = OneCycleLR(
    optimizer, 
    max_lr=0.0006, 
    steps_per_epoch=len(train_loader), 
    epochs=num_epochs, 
    pct_start=0.3, 
    div_factor=10, 
    final_div_factor=100
)

# 2. Tracking for Performance Validation [cite: 19]
trigger_times = 0
best_f1 = 0.0
unfrozen = False 
print(f"🚀 Starting High-Precision Training on {device}...")

for epoch in range(num_epochs):
    # --- LATENT UNFREEZING: Final 5% Accuracy Boost ---
    if epoch >= 35 and not unfrozen:
        print("🔓 Unfreezing Student Model for deep domain adaptation...")
        for param in student_model.parameters():
            param.requires_grad = True
        unfrozen = True
    
    student_model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for i, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)
        
        # 1. Ensemble Teacher Guidance (DenseNet & Swin) [cite: 32, 41]
        with torch.no_grad():
            t1_logits, t1_feats = teacher_dense(images)
            t2_logits, t2_feats = teacher_swin(images)
        
        # 2. Student Update [cite: 50]
        optimizer.zero_grad()
        s_logits, s_feats = student_model(images)
        
        # Multi-Teacher Distill Loss: Focal + Logit Distill + Feature Alignment [cite: 42, 54]
        loss = criterion(
            s_logits, 
            labels, 
            [t1_logits, t2_logits], 
            s_features=s_feats, 
            t_features_list=[t1_feats, t2_feats]
        )
        
        loss.backward()
        optimizer.step()
        scheduler.step() # Step per batch for OneCycleLR stability
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{running_loss/(i+1):.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.6f}"})
        
    # --- CLEAN METRIC LOGGING ---
    v_loss, v_acc, v_f1, v_rec = validate(student_model, val_loader, device)
    
    print(f"📊 Epoch {epoch+1} Summary:")
    print(f"Accuracy: {v_acc:.4f} | Recall: {v_rec:.4f} | Macro-F1: {v_f1:.4f} | Loss: {v_loss:.4f}")
    
    # Early Stopping based on Macro-F1 to handle class imbalance [cite: 142, 153]
    if v_f1 > best_f1:
        best_f1 = v_f1
        trigger_times = 0
        torch.save(student_model.state_dict(), 'best_student_model_aligned_final.pth')
        print(f"⭐ New Best Model Saved (F1: {best_f1:.4f})")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"🛑 Early Stopping triggered (Patience {patience} reached)[cite: 142].")
            break

print(f"\n✅ Training Finished. Final Best Macro-F1: {best_f1:.4f}")